# Full inference with contour narration and speaker association
Run top to bottom using godfather.venv. Local Magi + Paddle + YOLO, the trained character crop checkpoint, and the tested contour-narrator v2 rules. No narration classifier, DeBERTa, DINO or tail label override.

Pages → OCR/artifact deletion → overlap concatenation → contour SPOKEN/NARRATED → member character/entity association → grouped transcript.

Character identities are sequence-local E1, E2, ... . Missing/conflicting speakers remain unresolved and flagged. Reuse compatible caches; create missing stages locally. Input is a full 3-page sequence or one selected page. See `PIPELINE_CONTOUR_FULL_INFERENCE.md` for the Mermaid flow and output contracts.

## Flow
```mermaid
flowchart TD
    A[Ordered page images] --> B[Local Magi detection and OCR]
    B --> C[Local Paddle OCR on Magi text boxes]
    C --> D[Select OCR and delete agreement below 0.4 or SFX]
    D --> E[Surviving text IDs and original reading order]
    A --> F[Local YOLO bubble candidates]
    E --> G[Match bubbles to surviving Magi texts]
    F --> G
    G --> H[Check individual boundaries and faint retry]
    H --> I[Overlapping groups with an unresolved member]
    I --> J[Concatenate member OCR in Magi order]
    J --> K[Combined YOLO crop with 20 percent padding]
    K --> L[Extract white or solid background regions]
    L --> M{Confident region and shape?}
    M -->|No| N[Faint-edge barrier retry]
    M -->|Yes| O[Measure visible curves and box sides]
    N --> O
    O --> P{Substantial smooth white arc?}
    P -->|Yes| Q[SPOKEN]
    P -->|No| R{Visible perpendicular box sides?}
    R -->|Yes| S[NARRATED]
    R -->|No| T{Meaningful curved or sharp white edges?}
    T -->|Yes| Q
    T -->|No| U[NARRATED fallback]
    U --> V[Flag unclear extraction or conflicting regions]
    B --> W[Character crops and original text-character links]
    A --> X[Original RGB character crops]
    W --> Y[Fine-tuned last two crop encoder blocks]
    X --> Y
    Y --> Z[Duplicate cleanup and cross-page entity matching]
    Z --> AA[Original member text-to-entity candidates]
    Q --> AB[Resolve one speaker for each group]
    S --> AB
    U --> AB
    V --> AB
    AA --> AB
    AB --> AC{Group is NARRATED?}
    AC -->|Yes| AD[Speaker NARRATION]
    AC -->|No| AE{Distinct associated entities}
    AE -->|One| AF[Use entity ID; flag partial links]
    AE -->|Zero or multiple| AG[Unresolved speaker and review]
    AD --> AH[Annotated viewer and grouped transcript]
    AF --> AH
    AG --> AH
    AH --> AI[Original group and per-box audits]
    AH --> AJ[Local Gemma 3 4B IT cleanup per text entry]
    AJ --> AK[Cleaned labels-schema transcript and change audit]
    AH --> AL[Preserved raw transcript]
```

[Practical README and run guide](README_CONTOUR_FULL_INFERENCE.md) | [All fine-tuning experiments](../FINETUNING-EXPERMENTS/README.md)



## 1. Imports and offline execution

Import Torch before Paddle, as required by this environment. Model loading uses local files or the existing local Hugging Face cache.

In [1]:
# Set offline mode before importing Hugging Face model loaders.
import os
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK'] = 'True'
import torch  # Must precede Paddle on this Windows installation.
import paddle
import gc
import copy
import hashlib
import json
import sys
import unicodedata
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
try:
    import ipywidgets as widgets
except ImportError:
    widgets=None
from IPython.display import display
from PIL import Image
from transformers import AutoModel, AutoConfig
from paddleocr import PaddleOCR

print('Python:', sys.executable)
print('Torch:', torch.__version__, '| Paddle:', paddle.__version__)

import importlib.util
import inspect
import types
from scipy.optimize import linear_sum_assignment

# Load a local Python file under a unique module name: avoid collisions with other config/model files.
def load_local_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

import math
import cv2


C:\Users\ESHAAN\BYTE\godfather.venv\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
W1013 13:37:53.540000 33960 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


Python: C:\Users\ESHAAN\BYTE\godfather.venv\Scripts\python.exe
Torch: 2.14.0+cu126 | Paddle: 3.2.0


## 2. Sequence, local checkpoints and contour settings
Keep the original validated character checkpoint and threshold (epoch 20, cosine 0.82). Magi's detector and association head are original unless ASSOCIATION_HEAD_PATH is explicitly set. The last two fine-tuned crop-ViT blocks are used for character embeddings only. Contour settings come from the v2 helper and are editable below. Models are loaded only on cache misses; no downloads are performed.


In [2]:
MAGI_ROOT = Path(r'C:\Users\ESHAAN\BYTE\Applied-Task\Magi')
MODEL_PATH = Path(r'C:\MODELS\magiv2')
SPLIT = 'development'
SEQUENCE_ID = 'seq_2fae297a18af2a57'
LANGUAGE = 'en'
PREFERRED_OCR = 'paddle'
PAGE_NUMBER = None  # Set None for the full sequence; otherwise select this dataset page.
EXPERIMENT_ROOT = MAGI_ROOT / 'FINETUNING-EXPERMENTS/NarrationModelwBubbles'
CHECKPOINT_PATH = MAGI_ROOT /"FINETUNING-EXPERMENTS"/ 'fineTuningMagiCharSimHeads2' / 'magi_crop_last_2_blocks.pt'
ENTITY_THRESHOLD = 0.82  # Clustering threshold selected on labeled validation data before inference.
THRESHOLD_CHECKPOINT_EPOCH = 20
CHARACTER_CLEANUP_SETTINGS = dict(min_iou=0.90, min_cosine=0.95,
                                  algorithm='contained_direct_outer_match_v1')
# Bump after replacing model weights without changing their file metadata.
MODEL_REVISION = 'local-magiv2-1'
FORCE = dict(magi_raw=False, embeddings=False, paddle_ocr=False,
             ocr_selection=False, character_cleanup=False, entities=False, text_entities=False)
# Bump the affected version when editing a stage's algorithm.
STAGE_VERSIONS = dict(magi_raw=1, embeddings=1, paddle_ocr=1,
                     ocr_selection=1, character_cleanup=1, entities=1, text_entities=1)
PADDLE_SETTINGS = dict(lang=LANGUAGE, device='gpu:0', ocr_version='PP-OCRv5',
                      use_doc_orientation_classify=False, use_doc_unwarping=False,
                      use_textline_orientation=False)
SELECTOR_SETTINGS = dict(prefer=PREFERRED_OCR, check='unicode_text_validity_v1')
OCR_AGREEMENT_DELETE_BELOW = 0.4  # Strictly below; missing agreement is kept.
FORCE['ocr_agreement'] = False
STAGE_VERSIONS['ocr_agreement'] = 1

# Optional: apply the trained text-character head before Magi makes associations.
# Leave None to retain the original Magi head and reuse its compatible raw inference cache.
ASSOCIATION_HEAD_PATH = None
# ASSOCIATION_HEAD_PATH = MAGI_ROOT / 'FINETUNING-EXPERMENTS/Manga109Association/checkpoints/best_association_head.pt'

# Paddle weights must already be downloaded. Change these paths if your local models live elsewhere.
PADDLE_DET_PATH = Path.home() / '.paddlex/official_models/PP-OCRv5_server_det'
PADDLE_REC_PATH = Path.home() / '.paddlex/official_models/en_PP-OCRv5_mobile_rec'
PADDLE_LOCAL_SETTINGS = dict(text_detection_model_name='PP-OCRv5_server_det',
    text_recognition_model_name='en_PP-OCRv5_mobile_rec',
    text_detection_model_dir=str(PADDLE_DET_PATH), text_recognition_model_dir=str(PADDLE_REC_PATH))


DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
BUBBLE_MODEL=Path(r'C:/MODELS/manga_bubbles/bubbles_detect.pt')
DETECTION=dict(imgsz=1024,confidence=.25)
PIPELINE_ROOT=MAGI_ROOT/'paddleOCR-Magi'
CONTOUR_MODULE_PATH=PIPELINE_ROOT/'contour_narrator2.py'
CONTOUR_GROUPING_PATH=PIPELINE_ROOT/'contour_grouping.py'
contour_model=load_local_module('contour_full_inference_rules',CONTOUR_MODULE_PATH)
CONTOUR_SETTINGS=dict(contour_model.CHECK)
REGION_SETTINGS=dict(contour_model.REGION)
FAINT_REGION_SETTINGS=dict(contour_model.FAINT_REGION)
GROUP_FAINT_SETTINGS=dict(contour_model.contours.FAINT)
# Optional threshold edits belong here, before the stages below.
# REGION_SETTINGS['curve_turn_min_deg']=20
contour_model.CHECK=CONTOUR_SETTINGS
contour_model.REGION=REGION_SETTINGS
contour_model.FAINT_REGION=FAINT_REGION_SETTINGS
contour_model.contours.FAINT=GROUP_FAINT_SETTINGS
for name in ['contour_bubbles','contour_roles','contour_speakers']:
    FORCE[name]=False
    STAGE_VERSIONS[name]=1


## 3. Load sequence pages and discover caches

Pages are sorted using the existing dataset convention. Grayscale RGB feeds Magi; original RGB feeds the fine-tuned character crop encoder. Missing stage caches are built below.

In [3]:
# Ignore whitespace accidentally pasted into dataset settings.
SPLIT = str(SPLIT).strip()
SEQUENCE_ID = str(SEQUENCE_ID).strip()
SEQUENCE_ROOT = MAGI_ROOT / 'TaskData' / SPLIT / 'images' / SEQUENCE_ID
SHARED_CACHE_DIR = MAGI_ROOT / 'paddleOCR-Magi' / 'cache_inference' / SPLIT / SEQUENCE_ID
PAGE_NUMBER=None if PAGE_NUMBER is None else int(PAGE_NUMBER)
CACHE_SCOPE='sequence' if PAGE_NUMBER is None else f'page_{PAGE_NUMBER:02d}'
CACHE_DIR=PIPELINE_ROOT/'cache_contour_full_inference'/SPLIT/SEQUENCE_ID/CACHE_SCOPE
READ_CACHE_DIRS=list(dict.fromkeys([EXPERIMENT_ROOT/'cache_selected_ocr/full_pipeline'/SPLIT/SEQUENCE_ID/CACHE_SCOPE,
    EXPERIMENT_ROOT/'cache_selected_ocr/full_pipeline'/SPLIT/SEQUENCE_ID/'sequence',SHARED_CACHE_DIR]))
assert SEQUENCE_ROOT.is_dir(), f'Missing sequence: {SEQUENCE_ROOT}'
assert MODEL_PATH.is_dir(), f'Missing original model: {MODEL_PATH}'
assert CHECKPOINT_PATH.is_file(), f'Missing two-block checkpoint: {CHECKPOINT_PATH}'
PAGE_FILES = sorted(SEQUENCE_ROOT.glob('*.png'))
if PAGE_NUMBER is not None:
    PAGE_FILES = [p for p in PAGE_FILES if p.name == f'{PAGE_NUMBER:02d}.png']
assert PAGE_FILES, 'No PNG pages found'
chapter_pages, crop_pages = [], []
for path in PAGE_FILES:
    with Image.open(path) as image:
        chapter_pages.append(np.array(image.convert('L').convert('RGB')))
        crop_pages.append(np.array(image.convert('RGB')))  # Match the fine-tuning crop input path.
print('Sequence:', SEQUENCE_ID, '| pages:', [p.name for p in PAGE_FILES])
print('Cache:', CACHE_DIR)

assert 1 <= len(PAGE_FILES) <= 3, 'Supply one to three ordered pages.'
if PAGE_NUMBER is None:
    assert len(PAGE_FILES) == 3, 'Full sequence mode expects three pages; select PAGE_NUMBER for one page.'
assert BUBBLE_MODEL.is_file(), f'Missing local YOLO: {BUBBLE_MODEL}'
if ASSOCIATION_HEAD_PATH is not None:
    ASSOCIATION_HEAD_PATH = Path(ASSOCIATION_HEAD_PATH)
    assert ASSOCIATION_HEAD_PATH.is_file()
    # Keep changed association-head predictions separate from the original inference cache.
    head_digest = hashlib.sha256(ASSOCIATION_HEAD_PATH.read_bytes()).hexdigest()[:12]
    CACHE_DIR = CACHE_DIR / ('association_' + head_digest)
OUTPUT_DIR=CACHE_DIR/'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Existing stage caches:', [p.name for p in CACHE_DIR.glob('*.json')])
print('Full-pipeline exports:', OUTPUT_DIR)


Sequence: seq_2fae297a18af2a57 | pages: ['01.png', '02.png', '03.png']
Cache: C:\Users\ESHAAN\BYTE\Applied-Task\Magi\paddleOCR-Magi\cache_contour_full_inference\development\seq_2fae297a18af2a57\sequence
Existing stage caches: []
Full-pipeline exports: C:\Users\ESHAAN\BYTE\Applied-Task\Magi\paddleOCR-Magi\cache_contour_full_inference\development\seq_2fae297a18af2a57\sequence\outputs


## 4. Cache helpers and checkpoint checks

A cache is reused only when its input/settings signature and data checksum agree. Original character checkpoint validation is kept. Training notebooks are never executed here.

In [4]:
def json_ready(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(v) for v in value]
    return value


def digest(data):
    text = json.dumps(json_ready(data), sort_keys=True, ensure_ascii=True, allow_nan=False)
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


def file_hash(path):
    with path.open('rb') as file:
        return hashlib.file_digest(file, 'sha256').hexdigest()


def write_json(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(json_ready(data), ensure_ascii=True,
                                    indent=2, allow_nan=False), encoding='utf-8')
    temporary.replace(path)


def model_fingerprint():
    files = []
    for path in sorted(MODEL_PATH.rglob('*')):
        if path.is_file() and path.suffix in {'.json', '.py', '.safetensors', '.bin', '.pt', '.pth'}:
            stat = path.stat()
            files.append(dict(name=str(path.relative_to(MODEL_PATH)), size=stat.st_size,
                              mtime_ns=stat.st_mtime_ns,
                              sha256=file_hash(path) if path.suffix in {'.json', '.py'} else None))
    return dict(path=str(MODEL_PATH), revision=MODEL_REVISION, files=files)


PACKAGES = {name: version(name) for name in ['torch', 'torchvision', 'transformers',
            'paddlepaddle-gpu', 'paddleocr', 'paddlex', 'numpy', 'pillow']}
PAGE_METADATA = [dict(name=p.name, sha256=file_hash(p), width=image.shape[1], height=image.shape[0])
                 for p, image in zip(PAGE_FILES, chapter_pages)]
INPUTS = dict(sequence_id=SEQUENCE_ID, split=SPLIT, pages=PAGE_METADATA,
              preprocessing='grayscale_then_RGB')
MODEL_INFO = model_fingerprint()
if ASSOCIATION_HEAD_PATH is not None:
    MODEL_INFO['association_head_sha256'] = file_hash(ASSOCIATION_HEAD_PATH)
checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=True)
assert checkpoint['version'] == 2 and checkpoint['settings']['trained_blocks'] == 2
assert checkpoint['best_epoch'] == THRESHOLD_CHECKPOINT_EPOCH, (
    'Checkpoint epoch changed. Set ENTITY_THRESHOLD and THRESHOLD_CHECKPOINT_EPOCH from its validation clustering run.')
for name, expected in checkpoint['base_model_hashes'].items():
    assert file_hash(MODEL_PATH / name) == expected, f'Original model changed: {name}'
CHECKPOINT_INFO = dict(path=str(CHECKPOINT_PATH), sha256=file_hash(CHECKPOINT_PATH),
                       epoch=checkpoint['best_epoch'], trained_blocks=2)
manifest = dict(schema_version=1, inputs=INPUTS, model=MODEL_INFO, packages=PACKAGES,
                crop_encoder=CHECKPOINT_INFO, stages={})
print('Crop checkpoint epoch:', checkpoint['best_epoch'], '| clustering threshold:', ENTITY_THRESHOLD)
MANIFEST_PATH = CACHE_DIR / 'manifest.json'
if MANIFEST_PATH.exists():
    try:
        old_manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
        if old_manifest.get('inputs') == INPUTS:
            manifest['stages'] = old_manifest.get('stages', {})
    except (ValueError, OSError):
        pass


def stage_ref(name):
    entry = manifest['stages'][name]
    return dict(signature=entry['signature'], data_hash=entry['data_hash'])


def cached_stage(name, settings, compute):
    signature = digest(dict(inputs=INPUTS, algorithm_version=STAGE_VERSIONS[name], settings=settings))
    path = CACHE_DIR / f'{name}.json'
    payload = None
    if path.exists() and not FORCE[name]:
        try:
            candidate = json.loads(path.read_text(encoding='utf-8'))
            if candidate['signature'] == signature and digest(candidate['data']) == candidate['data_hash']:
                payload = candidate
        except (ValueError, KeyError, OSError):
            pass
    if payload is None and not FORCE[name]:
        for root in READ_CACHE_DIRS:
            try:
                candidate=json.loads((root/f'{name}.json').read_text(encoding='utf-8'))
                if candidate['signature']==signature and digest(candidate['data'])==candidate['data_hash']:
                    payload=candidate
                    write_json(path,payload)
                    print(f'IMPORT {name} from {root}')
                    break
            except (OSError,ValueError,KeyError,TypeError):
                continue
    if payload is None and not FORCE[name] and name in {'magi_raw', 'embeddings', 'paddle_ocr'}:
        for source_root in READ_CACHE_DIRS:
            try:
                shared_path = source_root / f'{name}.json'
                shared_manifest_path = source_root / 'manifest.json'
                if shared_path.is_file() and shared_manifest_path.is_file():
                    shared_manifest = json.loads(shared_manifest_path.read_text(encoding='utf-8'))
                    shared = json.loads(shared_path.read_text(encoding='utf-8'))
                    shared_inputs = shared_manifest['inputs']
                    shared_settings = copy.deepcopy(settings)
                    if 'magi' in shared_settings:
                        shared_raw = json.loads((source_root / 'magi_raw.json').read_text(encoding='utf-8'))
                        expected_raw = digest(dict(inputs=shared_inputs, algorithm_version=STAGE_VERSIONS['magi_raw'],
                            settings=dict(model=MODEL_INFO, torch=PACKAGES['torch'], torchvision=PACKAGES['torchvision'],
                                          transformers=PACKAGES['transformers'], do_ocr=True)))
                        if shared_raw['signature'] == expected_raw and digest(shared_raw['data']) == shared_raw['data_hash']:
                            shared_settings['magi'] = dict(signature=shared_raw['signature'], data_hash=shared_raw['data_hash'])
                    expected_shared = digest(dict(inputs=shared_inputs, algorithm_version=STAGE_VERSIONS[name], settings=shared_settings))
                    indices = []
                    for page in PAGE_METADATA:
                        matches = [i for i,p in enumerate(shared_inputs['pages']) if p == page]
                        if len(matches) != 1:
                            break
                        indices.append(matches[0])
                    if (len(indices) == len(PAGE_METADATA) and shared['signature'] == expected_shared and
                            digest(shared['data']) == shared['data_hash']):
                        data = copy.deepcopy([shared['data'][i] for i in indices])
                        if name == 'paddle_ocr':
                            for page_index, records in enumerate(data):
                                for record in records:
                                    record['page_index'] = page_index
                        payload = dict(signature=signature, data_hash=digest(data),
                            created_at=datetime.now(timezone.utc).isoformat(), data=data)
                        write_json(path, payload)
                        print(f'IMPORT compatible {name} from original full pipeline')
            except (OSError,ValueError,KeyError,TypeError,IndexError):
                continue
            if payload is not None:break
    if payload is None:
        print(f'RUN   {name}')
        data = json_ready(compute())
        payload = dict(signature=signature, data_hash=digest(data),
                       created_at=datetime.now(timezone.utc).isoformat(), data=data)
        write_json(path, payload)
    else:
        print(f'CACHE {name}')
    manifest['stages'][name] = dict(file=path.name, signature=signature,
                                  data_hash=payload['data_hash'], settings=settings,
                                  algorithm_version=STAGE_VERSIONS[name], created_at=payload['created_at'])
    write_json(MANIFEST_PATH, manifest)
    return payload['data']

def function_digest(function):
    # Function source is added to new stages' cache keys, so notebook edits invalidate their results.
    # Ignore notebook execution filenames/line numbers; they change across kernel restarts.
    def describe(code):
        constants = [describe(value) if isinstance(value, types.CodeType) else repr(value)
                     for value in code.co_consts]
        return dict(bytecode=code.co_code.hex(), names=code.co_names,
                    variables=code.co_varnames, constants=constants)
    return digest(describe(inspect.unwrap(function).__code__))


Crop checkpoint epoch: 20 | clustering threshold: 0.82


## 5. Magi and character embedding functions

Original detections/associations are produced before patching the final two character crop-ViT blocks. The optional association checkpoint changes only the text-character matching head.

In [5]:
magi_model = None


def get_magi():
    global magi_model
    if magi_model is None:
        assert torch.cuda.is_available(), 'Torch CUDA unavailable'
        # Normal tensors allow the later checkpoint copy; only predictions use inference mode.
        with torch.inference_mode(False):
            config = AutoConfig.from_pretrained(str(MODEL_PATH), trust_remote_code=True, local_files_only=True)
            # The full checkpoint already contains the backbone: avoid timm downloading it again.
            config.detection_model_config.use_pretrained_backbone = False
            config.disable_ocr = False
            config.disable_crop_embeddings = False
            magi_model, info = AutoModel.from_pretrained(str(MODEL_PATH), config=config,
                trust_remote_code=True, local_files_only=True, output_loading_info=True)
            assert not info['missing_keys'] and not info.get('mismatched_keys'), info
            magi_model = magi_model.cuda().eval()
            magi_model.requires_grad_(False)
            if ASSOCIATION_HEAD_PATH is not None:
                association = torch.load(ASSOCIATION_HEAD_PATH, map_location='cpu', weights_only=True)
                magi_model.text_character_matching_head.load_state_dict(association['state_dict'], strict=True)
    return magi_model


def predict_magi():
    model = get_magi()
    with torch.inference_mode():
        return model.do_chapter_wide_prediction(
            chapter_pages, {'images': [], 'names': []}, use_tqdm=True, do_ocr=True)


def predict_embeddings():
    boxes = [result['characters'] for result in magi_raw_results]
    if not any(boxes):
        return [[] for _ in boxes]
    model = get_magi()
    last = len(model.crop_embedding_model.encoder.layer) - 1
    assert last >= 1
    prefixes = (f'crop_embedding_model.encoder.layer.{last-1}.',
                f'crop_embedding_model.encoder.layer.{last}.')
    parameters = dict(model.named_parameters())
    expected_names = {name for name in parameters if name.startswith(prefixes)}
    assert tuple(checkpoint['trainable_prefixes']) == prefixes
    assert set(checkpoint['state_dict']) == expected_names
    # Patch only the two crop blocks after original Magi prediction is complete.
    with torch.no_grad():
        for name, tensor in checkpoint['state_dict'].items():
            assert tensor.shape == parameters[name].shape
            parameters[name].copy_(tensor.to(parameters[name].device))
    model.eval()
    with torch.inference_mode():
        embeddings = model.predict_crop_embeddings(crop_pages, boxes, mask_ratio=0.0, batch_size=32)
        return [torch.nn.functional.normalize(x, p=2, dim=1) if len(x) else x for x in embeddings]


def validate_magi(results):
    assert len(results) == len(chapter_pages), 'Magi page count mismatch'
    for result in results:
        assert len(result['texts']) == len(result['ocr']), 'Text/OCR count mismatch'
        assert len(result['characters']) == len(result['character_cluster_labels']), 'Character/cluster count mismatch'
        for text_index, char_index in result['text_character_associations']:
            assert 0 <= text_index < len(result['texts']), 'Invalid text association index'
            assert 0 <= char_index < len(result['characters']), 'Invalid character association index'


## 6. Reuse or create Magi and character caches

Cache hits avoid loading Magi. On misses, run the local model, cache each completed stage, and release its GPU memory.

In [6]:
try:
    magi_raw_results = cached_stage('magi_raw',
        dict(model=MODEL_INFO, torch=PACKAGES['torch'], torchvision=PACKAGES['torchvision'],
             transformers=PACKAGES['transformers'], do_ocr=True), predict_magi)
    validate_magi(magi_raw_results)
    embeddings_per_page = cached_stage('embeddings',
        dict(magi=stage_ref('magi_raw'), model=MODEL_INFO, checkpoint=CHECKPOINT_INFO,
             crop_preprocessing='original_RGB', mask_ratio=0.0, batch_size=32, normalization='L2',
             torch=PACKAGES['torch'], transformers=PACKAGES['transformers']), predict_embeddings)
finally:
    magi_model = None
    gc.collect()
    torch.cuda.empty_cache()

assert len(embeddings_per_page) == len(magi_raw_results)
for result, embeddings in zip(magi_raw_results, embeddings_per_page):
    assert len(embeddings) == len(result['characters']), 'Character/embedding count mismatch'
    assert np.isfinite(np.asarray(embeddings)).all(), 'Nonfinite embeddings'
per_page_results = copy.deepcopy(magi_raw_results)
print('Text counts:', [len(r['texts']) for r in per_page_results])


RUN   magi_raw


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
100%|██████████| 1/1 [00:19<00:00, 19.44s/it]


RUN   embeddings
Text counts: [11, 9, 8]


## 7. Paddle OCR functions

Recognize each original text crop. Invalid crops and per-box errors keep placeholders, allowing Magi OCR fallback without shifting text indices.

In [7]:
def crop_box(image, box):
    values = np.asarray(box, dtype=float)
    if values.shape != (4,) or not np.isfinite(values).all():
        return None, None
    height, width = image.shape[:2]
    x1, y1, x2, y2 = values
    clipped = [int(np.clip(np.floor(x1), 0, width)), int(np.clip(np.floor(y1), 0, height)),
               int(np.clip(np.ceil(x2), 0, width)), int(np.clip(np.ceil(y2), 0, height))]
    x1, y1, x2, y2 = clipped
    if x2 <= x1 or y2 <= y1:
        return None, clipped
    return np.ascontiguousarray(image[y1:y2, x1:x2]), clipped


def predict_paddle():
    output = [[] for _ in chapter_pages]
    if not any(result['texts'] for result in magi_raw_results):
        return output
    assert paddle.is_compiled_with_cuda() and paddle.device.cuda.device_count() > 0, 'Paddle CUDA unavailable'
    assert PADDLE_DET_PATH.is_dir() and PADDLE_REC_PATH.is_dir(), 'Set the local Paddle model paths in cell 2.'
    # Explicit local directories prevent downloading detector/recognizer models.
    engine = PaddleOCR(**PADDLE_SETTINGS, **PADDLE_LOCAL_SETTINGS)
    try:
        for page_index, (image, result) in enumerate(zip(chapter_pages, magi_raw_results)):
            for text_index, box in enumerate(result['texts']):
                crop, clipped = crop_box(image, box)
                record = dict(page_index=page_index, text_index=text_index, text_id=f'T{text_index+1}',
                              box=box, crop_box=clipped, text='', lines=[], status='invalid_crop', error=None)
                if crop is not None:
                    try:
                        for prediction in engine.predict(input=crop):
                            texts, scores = prediction['rec_texts'], prediction['rec_scores']
                            assert len(texts) == len(scores), 'Paddle text/score count mismatch'
                            for text, score in zip(texts, scores):
                                record['lines'].append(dict(text=str(text), confidence=float(score)))
                        record['text'] = ' '.join(line['text'] for line in record['lines'])
                        record['status'] = 'text' if record['text'].strip() else 'empty'
                    except Exception as error:
                        record.update(text='', lines=[], status='error', error=str(error))
                        print(f'PG{page_index+1}:T{text_index+1} OCR error: {error}')
                output[page_index].append(record)
    finally:
        del engine
        gc.collect()
        paddle.device.cuda.empty_cache()
    return output


## 8. Reuse or create Paddle OCR cache

Existing compatible OCR is reused. A missing cache runs Paddle on the original text boxes; no new text detector boxes replace Magi boxes.

In [8]:
paddle_ocr_results = cached_stage('paddle_ocr',
    dict(magi=stage_ref('magi_raw'), paddle=PADDLE_SETTINGS, local_models=PADDLE_LOCAL_SETTINGS,
         packages={k: PACKAGES[k] for k in ['paddlepaddle-gpu', 'paddleocr', 'paddlex', 'numpy', 'pillow']}),
    predict_paddle)
print('Paddle records:', sum(len(page) for page in paddle_ocr_results))
print('Cached OCR errors:', sum(r['status'] == 'error' for page in paddle_ocr_results for r in page))


RUN   paddle_ocr


C:\Users\ESHAAN\AppData\Local\Temp\ipykernel_33960\844908468.py:22: UserWarning: `lang` and `ocr_version` will be ignored when model names or model directories are not `None`.
  engine = PaddleOCR(**PADDLE_SETTINGS, **PADDLE_LOCAL_SETTINGS)
Creating model: ('PP-OCRv5_server_det', 'C:\\Users\\ESHAAN\\.paddlex\\official_models\\PP-OCRv5_server_det', None)
Creating model: ('en_PP-OCRv5_mobile_rec', 'C:\\Users\\ESHAAN\\.paddlex\\official_models\\en_PP-OCRv5_mobile_rec', None)


Paddle records: 28
Cached OCR errors: 0


## 9. OCR selection rules

Use the existing text-validity checks. Prefer Paddle when both outputs pass; retain source, confidence lines and review flags.

In [9]:
def text_validity(text):
    if text is None:
        return dict(passes=False, reason='empty', text='')
    if not isinstance(text, str):
        return dict(passes=False, reason='not_a_string', text='')
    normalized = ' '.join(text.split())
    if not normalized:
        reason = 'empty'
    elif '\ufffd' in text or any(unicodedata.category(c) in {'Cc', 'Cs'} and c not in '\n\r\t' for c in text):
        reason = 'invalid_characters'
    elif not any(c.isalnum() for c in normalized):
        reason = 'no_letters_or_digits'
    else:
        reason = 'valid_text'
    return dict(passes=reason == 'valid_text', reason=reason, text=normalized)


def select_ocr(paddle_text, magi_text, prefer='paddle'):
    assert prefer in {'paddle', 'magi'}, 'Choose paddle or magi'
    checks = {'paddle': text_validity(paddle_text), 'magi': text_validity(magi_text)}
    passing = [source for source, check in checks.items() if check['passes']]
    nonempty = [source for source, check in checks.items() if check['text']]
    if len(passing) == 1:
        source, reason = passing[0], 'only_valid_output'
    elif len(passing) == 2:
        source, reason = prefer, 'both_valid_preferred'
    elif len(nonempty) == 1:
        source, reason = nonempty[0], 'only_nonempty_needs_review'
    else:
        source = None
        reason = 'both_empty' if all(c['reason'] == 'empty' for c in checks.values()) else 'both_invalid'
    return dict(paddle_text=paddle_text, magi_text=magi_text, validity=checks,
                selected_text=checks[source]['text'] if source else '', selected_source=source,
                reason=reason, needs_review=not bool(passing),
                disagreement=len(passing) == 2 and checks['paddle']['text'].casefold() != checks['magi']['text'].casefold())


def select_sequence_ocr():
    assert len(paddle_ocr_results) == len(magi_raw_results), 'Paddle page count mismatch'
    selected = []
    for page_index, (magi, paddle_records) in enumerate(zip(magi_raw_results, paddle_ocr_results)):
        assert len(paddle_records) == len(magi['texts']) == len(magi['ocr']), 'OCR count mismatch'
        decisions = []
        for text_index, (box, record, magi_text) in enumerate(zip(magi['texts'], paddle_records, magi['ocr'])):
            assert record['page_index'] == page_index and record['text_index'] == text_index, 'OCR order mismatch'
            assert record['box'] == box, 'OCR crop mismatch'
            decision = select_ocr(record['text'], magi_text, PREFERRED_OCR)
            decision.update(page_index=page_index, text_index=text_index, text_id=f'T{text_index+1}', box=box,
                            paddle_lines=record['lines'], paddle_status=record['status'], paddle_error=record['error'])
            decisions.append(decision)
        selected.append(decisions)
    return selected


## 10. Select OCR and preserve box order

Both raw OCR outputs remain available. The selected OCR is used for narration context and final dialogue text.

OCR agreement filter: after selection, normalize both outputs to lowercase ASCII letters/digits and compare with SequenceMatcher. Agreement strictly below 0.4 deletes the box from dialogue/transcript exports and hides it in the default viewer. Missing agreement is kept. Original indices, feature inputs and full per-box audit records are retained. Enable `show deleted` in the viewer to inspect rejected crops.

Early clutter deletion checks both OCR outputs and selected text for SFX/SKX markers, URLs and fuzzy watermark phrases (threshold 0.85). Edit `CLUTTER_PHRASES` above the filter. Matching reasons and OCR source are recorded in `clutter_matches`. Deleted boxes are excluded before contour grouping.


In [ ]:
ocr_selection = cached_stage('ocr_selection',
    dict(magi=stage_ref('magi_raw'), paddle=stage_ref('paddle_ocr'), selector=SELECTOR_SETTINGS),
    select_sequence_ocr)
per_page_results = copy.deepcopy(magi_raw_results)
assert len(ocr_selection) == len(per_page_results), 'Selected OCR page count mismatch'
for page_index, (result, decisions) in enumerate(zip(per_page_results, ocr_selection)):
    assert len(decisions) == len(result['texts']), 'Selected OCR count mismatch'
    for text_index, decision in enumerate(decisions):
        assert decision['page_index'] == page_index and decision['text_index'] == text_index
        assert decision['box'] == result['texts'][text_index], 'Selected OCR box mismatch'
    result['magi_ocr'] = list(result['ocr'])
    result['ocr'] = [record['selected_text'] for record in decisions]
for raw, selected in zip(magi_raw_results, per_page_results):
    assert raw['texts'] == selected['texts'] and len(raw['texts']) == len(selected['ocr'])
display(pd.DataFrame([dict(page=r['page_index']+1, text_id=r['text_id'],
                          paddle=r['paddle_text'], magi=r['magi_text'], selected=r['selected_text'],
                          source=r['selected_source'], reason=r['reason'], review=r['needs_review'],
                          disagreement=r['disagreement']) for page in ocr_selection for r in page]))


# Early clutter filter: edit phrases/threshold here. Raw OCR and box IDs stay in the audit.
CLUTTER_PHRASES = ['manga stream', 'do not mirror', 'do not redistribute', 'read at mangastream']
CLUTTER_FUZZY_THRESHOLD = 0.85


def find_ocr_clutter(text):
    import re
    import unicodedata
    from difflib import SequenceMatcher
    text = unicodedata.normalize('NFKC', str(text or '')).casefold()
    words = re.findall(r'[a-z0-9]+', text)
    # Short markers are exact tokens, so legitimate SIX/SEX are not fuzzy matches.
    if any(word in {'sfx', 'skx'} for word in words):
        return 'SFX/SKX marker'
    compact = re.sub(r'\s+', '', text)
    if re.search(r'https?[:;/]+|www[.,]|(?:[a-z0-9-]+\.)+(?:com|net|org|co\.uk|io|me)(?:[/?:#]|$)', compact):
        return 'link/URL'
    # Normalize common OCR digit substitutions only for watermark matching.
    words = [word.translate(str.maketrans({'0':'o', '4':'a', '1':'i'})) for word in words]
    for phrase in CLUTTER_PHRASES:
        target = ''.join(re.findall(r'[a-z0-9]+', phrase))
        for size in range(1, len(phrase.split()) + 2):
            for start in range(len(words) - size + 1):
                window = ''.join(words[start:start + size])
                score = SequenceMatcher(None, target, window).ratio()
                if score >= CLUTTER_FUZZY_THRESHOLD:
                    return f'watermark: {phrase} (match={score:.2f})'
    return ''


# Preserve feature alignment; these deletion flags exclude boxes before contour processing.
def measure_ocr_agreement():
    import re
    from difflib import SequenceMatcher
    output=[]
    for decisions in ocr_selection:
        page=[]
        for decision in decisions:
            a=''.join(re.findall('[a-z0-9]',str(decision['magi_text'] or '').lower()))
            b=''.join(re.findall('[a-z0-9]',str(decision['paddle_text'] or '').lower()))
            agreement=SequenceMatcher(None,a,b).ratio() if a and b else None
            low_agreement=agreement is not None and agreement<OCR_AGREEMENT_DELETE_BELOW
            # Any OCR source containing SFX deletes the whole box, even with high agreement.
            contains_sfx=any('sfx' in str(decision.get(key) or '').casefold()
                             for key in ['magi_text','paddle_text','selected_text'])
            clutter_matches=[dict(source=key,reason=reason)
                for key in ['magi_text','paddle_text','selected_text']
                if (reason:=find_ocr_clutter(decision.get(key)))]
            reasons=[]
            if low_agreement: reasons.append(f'OCR agreement below {OCR_AGREEMENT_DELETE_BELOW}')
            if contains_sfx: reasons.append('OCR contains SFX')
            reasons.extend(f"{hit['source']}: {hit['reason']}" for hit in clutter_matches)
            page.append(dict(ocr_agreement=agreement,contains_sfx=contains_sfx,
                clutter_matches=clutter_matches,delete_ocr=bool(reasons),deletion_reason='; '.join(reasons)))
        output.append(page)
    return output

ocr_agreement = cached_stage('ocr_agreement',
    dict(selector=stage_ref('ocr_selection'),threshold=OCR_AGREEMENT_DELETE_BELOW,delete_sfx=True,
         clutter_phrases=CLUTTER_PHRASES,clutter_threshold=CLUTTER_FUZZY_THRESHOLD,
         clutter_algorithm=function_digest(find_ocr_clutter),
         algorithm=function_digest(measure_ocr_agreement)),measure_ocr_agreement)
display(pd.DataFrame([dict(page=p+1,text_id=f'T{i+1}',**record)
    for p,page in enumerate(ocr_agreement) for i,record in enumerate(page)]))


## 11. Remove duplicate character appearances

Reuse the existing containment + IoU + embedding-similarity rule. Redirect text-character links, remap character indices, and leave text indices unchanged.

In [11]:
def remove_duplicate_characters(results, embeddings, min_iou=0.90, min_cosine=0.95):
    """Remove contained, visually similar duplicates; preserve all text indices."""
    assert 0 <= min_iou <= 1 and -1 <= min_cosine <= 1
    assert len(results) == len(embeddings)
    cleaned_pages, cleaned_embeddings, audits = [], [], []
    for page_index, (raw, vectors) in enumerate(zip(results, embeddings)):
        boxes = np.asarray(raw['characters'], dtype=float).reshape(-1, 4)
        assert len(vectors) == len(boxes), 'Character/embedding count mismatch'
        assert len(raw['character_cluster_labels']) == len(boxes)
        areas = np.maximum(0, boxes[:, 2] - boxes[:, 0]) * np.maximum(0, boxes[:, 3] - boxes[:, 1])
        # Largest first. Equal areas prefer the earlier original character index.
        retained, representatives, decisions = [], {}, []
        for inner in sorted(range(len(boxes)), key=lambda i: (-areas[i], i)):
            representative = inner
            for outer in retained:
                a, b = boxes[inner], boxes[outer]
                if not np.isfinite(a).all() or not np.isfinite(b).all() or areas[inner] <= 0 or areas[outer] <= 0:
                    continue
                contained = a[0] >= b[0] and a[1] >= b[1] and a[2] <= b[2] and a[3] <= b[3]
                iou = float(areas[inner] / areas[outer]) if contained else 0.0
                if not contained or iou < min_iou:
                    continue
                x, y = np.asarray(vectors[inner], dtype=float), np.asarray(vectors[outer], dtype=float)
                assert x.ndim == y.ndim == 1 and x.shape == y.shape
                denominator = np.linalg.norm(x) * np.linalg.norm(y)
                if not np.isfinite(x).all() or not np.isfinite(y).all() or not np.isfinite(denominator) or denominator <= 0:
                    continue
                cosine = float(np.clip(np.dot(x, y) / denominator, -1, 1))
                if cosine >= min_cosine:
                    representative = outer
                    decisions.append(dict(removed_original_index=inner, retained_original_index=outer,
                                          smaller_box_coverage=1.0, iou=iou, cosine=cosine))
                    break
            representatives[inner] = representative
            if representative == inner:
                retained.append(inner)
        # Keep the original relative order of surviving character appearances.
        kept = sorted(retained)
        new_index = {old: new for new, old in enumerate(kept)}
        old_to_new = [new_index[representatives[i]] for i in range(len(boxes))]
        page = copy.deepcopy(raw)
        for key in ['characters', 'character_cluster_labels', 'character_names']:
            if key in raw:
                assert len(raw[key]) == len(boxes), f'Misaligned {key}'
                page[key] = [raw[key][i] for i in kept]
        links, seen = [], set()
        for text_index, char_index in raw['text_character_associations']:
            assert 0 <= text_index < len(raw['texts']) and 0 <= char_index < len(boxes)
            link = (text_index, old_to_new[char_index])
            if link not in seen:
                seen.add(link)
                links.append(list(link))
        page['text_character_associations'] = links
        assert page['texts'] == raw['texts'] and page['ocr'] == raw['ocr']
        cleaned_pages.append(page)
        cleaned_embeddings.append([vectors[i] for i in kept])
        audits.append(dict(page_index=page_index, kept_original_indices=kept,
                           original_to_cleaned_index=old_to_new, removals=decisions))
    return dict(pages=cleaned_pages, embeddings=cleaned_embeddings, audit=audits)


character_cleanup = cached_stage('character_cleanup',
    dict(magi=stage_ref('magi_raw'), embeddings=stage_ref('embeddings'),
         cleanup=CHARACTER_CLEANUP_SETTINGS),
    lambda: remove_duplicate_characters(magi_raw_results, embeddings_per_page,
        CHARACTER_CLEANUP_SETTINGS['min_iou'], CHARACTER_CLEANUP_SETTINGS['min_cosine']))
character_results = character_cleanup['pages']
character_embeddings_per_page = character_cleanup['embeddings']
validate_magi(character_results)
# Apply only character fields; preserve the selected OCR already in memory.
for result, cleaned in zip(per_page_results, character_results):
    for key in ['characters', 'character_cluster_labels', 'character_names', 'text_character_associations']:
        if key in cleaned:
            result[key] = copy.deepcopy(cleaned[key])
print('Character counts before:', [len(r['characters']) for r in magi_raw_results])
print('Character counts after: ', [len(r['characters']) for r in character_results])
display(pd.DataFrame([dict(page_index=page['page_index'], **decision)
                      for page in character_cleanup['audit'] for decision in page['removals']]))


RUN   character_cleanup
Character counts before: [7, 7, 8]
Character counts after:  [7, 7, 8]


""


## 12. Cross-page entity matching functions

Match page-local clusters to sequence entities using the strongest member cosine similarity. Each entity accepts at most one local cluster per page.

In [12]:
def page_clusters(result, page_index):
    groups = {}
    for char_index, label in enumerate(result['character_cluster_labels']):
        cluster_id = f'PG{page_index+1}:CL{int(label)+1}' if int(label) >= 0 else f'PG{page_index+1}:C{char_index+1}:unassigned'
        groups.setdefault(cluster_id, []).append(char_index)
    return groups


def clusterize(results, embeddings, threshold=0.50):
    assert -1 <= threshold <= 1, 'Cosine threshold must be between -1 and 1'
    vectors = [np.asarray(page, dtype=float) for page in embeddings]
    assert len(results) == len(vectors)
    for result, page_vectors in zip(results, vectors):
        assert len(result['characters']) == len(page_vectors), 'Embedding count mismatch'
    entities, decisions, char_to_entity = {}, [], {}
    for page_index, result in enumerate(results):
        clusters = page_clusters(result, page_index)
        candidates = []
        for cluster_id, indices in clusters.items():
            for entity_id, entity in entities.items():
                best_score, best_pair = -float('inf'), None
                for current in indices:
                    for member in entity['members']:
                        score = float(np.dot(vectors[page_index][current], vectors[member['page_index']][member['character_index']]))
                        if score > best_score:
                            best_score = score
                            best_pair = dict(current=f'PG{page_index+1}:C{current+1}', existing=member['character_id'])
                if best_score >= threshold:
                    candidates.append(dict(cluster_id=cluster_id, entity_id=entity_id, score=best_score, best_pair=best_pair))
        candidates.sort(key=lambda candidate: candidate['score'], reverse=True)
        accepted, used_entities = {}, set()
        for candidate in candidates:
            if candidate['cluster_id'] not in accepted and candidate['entity_id'] not in used_entities:
                accepted[candidate['cluster_id']] = candidate
                used_entities.add(candidate['entity_id'])
        for cluster_id, indices in clusters.items():
            match = accepted.get(cluster_id)
            entity_id = match['entity_id'] if match else f'E{len(entities)+1}'
            if entity_id not in entities:
                entities[entity_id] = dict(clusters=[], members=[])
            entities[entity_id]['clusters'].append(cluster_id)
            for char_index in indices:
                character_id = f'PG{page_index+1}:C{char_index+1}'
                entities[entity_id]['members'].append(dict(page_index=page_index, character_index=char_index,
                                                         character_id=character_id, cluster_id=cluster_id))
                char_to_entity[character_id] = entity_id
            decisions.append(dict(page_index=page_index, cluster_id=cluster_id, entity_id=entity_id,
                                  reason='matched' if match else 'new_entity',
                                  score=match['score'] if match else None, best_pair=match['best_pair'] if match else None))
    return dict(entities=entities, char_to_entity=char_to_entity, decisions=decisions)


## 13. Reuse or create entity identities

Use the existing validated clustering threshold and save the matching audit. Character identities are scoped to this sequence.

In [13]:
entity_results = cached_stage('entities',
    dict(characters=stage_ref('character_cleanup'), threshold=ENTITY_THRESHOLD,
         algorithm='sequential_best_member_one_cluster_per_entity_per_page'),
    lambda: clusterize(character_results, character_embeddings_per_page, ENTITY_THRESHOLD))
identified_char = entity_results['entities']
char_to_entity = entity_results['char_to_entity']
assert len(char_to_entity) == sum(len(r['characters']) for r in character_results)
print('Entities:', len(identified_char))
display(pd.DataFrame(entity_results['decisions']))


RUN   entities
Entities: 9


,page_index,cluster_id,entity_id,reason,score,best_pair
0,0,PG1:CL1,E1,new_entity,NaN,None
1,0,PG1:CL2,E2,new_entity,NaN,None
2,0,PG1:CL3,E3,new_entity,NaN,None
3,0,PG1:CL4,E4,new_entity,NaN,None
4,1,PG2:CL1,E5,new_entity,NaN,None
5,1,PG2:CL2,E3,matched,0.948633,"{'current': 'PG2:C4', 'existing': 'PG1:C6'}"
6,1,PG2:CL3,E6,new_entity,NaN,None
7,1,PG2:CL4,E7,new_entity,NaN,None
8,2,PG3:CL1,E8,new_entity,NaN,None
9,2,PG3:CL2,E2,matched,0.860615,"{'current': 'PG3:C3', 'existing': 'PG1:C2'}"


## 14. Text-to-entity mapping functions

Follow cleaned text → character → entity links. Missing links remain unassociated, and multiple entities remain ambiguous.

In [14]:
def map_text_entities(results, selections, entity_map, sequence_id, page_names):
    assert len(results) == len(selections) == len(page_names)
    records = []
    for page_index, (result, decisions) in enumerate(zip(results, selections)):
        assert len(result['texts']) == len(result['ocr']) == len(decisions)
        links_by_text = {}
        for text_index, char_index in result['text_character_associations']:
            assert 0 <= text_index < len(result['texts']) and 0 <= char_index < len(result['characters'])
            links_by_text.setdefault(text_index, []).append(char_index)
        for text_index, (box, text, decision) in enumerate(zip(result['texts'], result['ocr'], decisions)):
            assert decision['page_index'] == page_index and decision['text_index'] == text_index and decision['box'] == box
            assert decision['selected_text'] == text
            links = []
            for char_index in dict.fromkeys(links_by_text.get(text_index, [])):
                char_id = f'PG{page_index+1}:C{char_index+1}'
                links.append(dict(character_index=char_index, character_id=char_id, entity_id=entity_map[char_id]))
            entity_ids = list(dict.fromkeys(link['entity_id'] for link in links))
            records.append(dict(sequence_id=sequence_id, page_index=page_index, page_file=page_names[page_index],
                                text_index=text_index, text_id=f'T{text_index+1}', box=box, text=text,
                                entity_id=entity_ids[0] if len(entity_ids) == 1 else None,
                                entity_ids=entity_ids, character_links=links,
                                association_status='unassociated' if not links else 'linked' if len(entity_ids) == 1 else 'ambiguous',
                                ocr_source=decision['selected_source'], ocr_reason=decision['reason'],
                                needs_review=decision['needs_review'] or len(entity_ids) > 1,
                                disagreement=decision['disagreement']))
    return records


## 15. Reuse or create speaker candidates

Save one candidate record for every original text box, including empty OCR. Narration will override the final speaker, while preserving these raw candidates.

In [15]:
text_entities = cached_stage('text_entities',
    dict(magi=stage_ref('magi_raw'), characters=stage_ref('character_cleanup'),
         selector=stage_ref('ocr_selection'), entities=stage_ref('entities')),
    lambda: map_text_entities(per_page_results, ocr_selection, char_to_entity, SEQUENCE_ID,
                             [path.name for path in PAGE_FILES]))
assert len(text_entities) == sum(len(r['texts']) for r in magi_raw_results)
columns = ['page_index', 'text_id', 'text', 'entity_id', 'association_status',
           'ocr_source', 'needs_review', 'disagreement']
display(pd.DataFrame(text_entities, columns=columns))

for record in text_entities:
    record['box_id']=f"PG{int(PAGE_FILES[record['page_index']].stem)}:T{record['text_index']+1}"
    record.update(ocr_agreement[record['page_index']][record['text_index']])
    decision=ocr_selection[record['page_index']][record['text_index']]
    record.update(magi_text=decision['magi_text'],paddle_text=decision['paddle_text'])
# Keep these candidates immutable; final group speakers are resolved later.
raw_text_entities=copy.deepcopy(text_entities)


RUN   text_entities


,page_index,text_id,text,entity_id,association_status,ocr_source,needs_review,disagreement
0,0,T1,MANGA STREAM DO NOT MIRROR,E3,linked,paddle,False,True
1,0,T2,GOT YOUR TOMATO DEVIL RIGHT HERE.,E3,linked,paddle,False,False
2,0,T3,"IT REVIVES FROM ITS SEEDS, SO YOU SHOULD BURN IT.",E3,linked,paddle,False,True
3,0,T4,"GOOD WORK, DENJI.",E1,linked,paddle,False,False
4,0,T5,THIS GUY'S CORPSE'LL FETCH A PRETTY PENNY ON T...,E1,linked,paddle,False,False
5,0,T6,THE REWARD'S 400 GRAND.,E1,linked,paddle,False,False
6,0,T7,Though you look at the bottom-of-the-the-the-a...,E1,linked,magi,False,False
7,0,T8,THANKS !!,E3,linked,paddle,False,False
8,0,T9,I'M TAKING OHT 17O GRAND tO COVER YOUR LOAN AN...,NaN,unassociated,paddle,False,True
9,0,T10,AND THEN THERE'S THE COMMISSION FOR REFERRING ...,NaN,unassociated,paddle,False,True


## 16. Reuse YOLO bubbles or detect missing pages
Reuse matching bubble JSON or standalone test caches using page hash, local weights hash, image size and confidence. Keep all detections, including candidates discarded from old visual features. Missing pages run local YOLO only; no DETR bubble embeddings are needed.


In [16]:
BUBBLE_SHA256=file_hash(BUBBLE_MODEL)
BUBBLE_CACHE_ROOTS=[EXPERIMENT_ROOT/'cache_selected_ocr/bubbles'/SPLIT/SEQUENCE_ID,
                    EXPERIMENT_ROOT/'cache_selected_ocr/bubbles/full_pipeline'/SEQUENCE_ID]
LEGACY_TEST_CACHES=[PIPELINE_ROOT/'cache_test_contour_narrator2',PIPELINE_ROOT/'cache_test_contour_narrator']


def find_cached_bubbles(page_index):
    page=PAGE_METADATA[page_index]
    for root in BUBBLE_CACHE_ROOTS:
        for path in sorted(root.rglob('bubbles.json'))+sorted(root.rglob('page.pt')):
            try:
                saved=torch.load(path,map_location='cpu',weights_only=True) if path.suffix=='.pt' else json.loads(path.read_text(encoding='utf-8'))
                identity=saved['identity'];settings=identity['settings']
                if (saved['page']['sha256']==page['sha256']
                    and (saved['page']['width'],saved['page']['height'])==(page['width'],page['height'])
                    and identity['detector_sha256']==BUBBLE_SHA256
                    and settings['bubble_imgsz']==DETECTION['imgsz']
                    and settings['bubble_confidence']==DETECTION['confidence']):
                    return [dict(box=b['box'],confidence=b['confidence']) for b in saved['bubbles']],str(path)
            except (OSError,KeyError,ValueError,TypeError,RuntimeError):continue
    # Standalone caches have a deterministic name based on the same identity plus a payload checksum.
    key=digest(dict(page=page['sha256'],weights=BUBBLE_SHA256,settings=DETECTION))[:20]
    for root in LEGACY_TEST_CACHES:
        path=root/f'bubbles_{key}.json'
        try:
            saved=json.loads(path.read_text(encoding='utf-8'))
            if digest(saved['data'])==saved['data_hash']:return saved['data'],str(path)
        except (OSError,KeyError,ValueError,TypeError,RuntimeError):continue
    return None,None


def predict_bubbles():
    pages=[];missing=[]
    for page_index in range(len(PAGE_FILES)):
        candidates,source=find_cached_bubbles(page_index)
        pages.append(dict(candidates=candidates,source=source))
        if candidates is None:missing.append(page_index)
    if missing:
        from ultralytics import YOLO
        detector=YOLO(str(BUBBLE_MODEL))
        try:
            for page_index in missing:
                prediction=detector.predict(source=Image.fromarray(crop_pages[page_index]),
                    imgsz=DETECTION['imgsz'],conf=DETECTION['confidence'],
                    device=0 if DEVICE=='cuda' else 'cpu',verbose=False)[0]
                height,width=crop_pages[page_index].shape[:2];candidates=[]
                for box,score in zip(prediction.boxes.xyxy.cpu().tolist(),prediction.boxes.conf.cpu().tolist()):
                    a,b,c,d=box
                    bounds=[max(0,math.floor(a)),max(0,math.floor(b)),min(width,math.ceil(c)),min(height,math.ceil(d))]
                    if bounds[2]>bounds[0] and bounds[3]>bounds[1]:candidates.append(dict(box=bounds,confidence=float(score)))
                candidates.sort(key=lambda b:(b['box'][1],-b['box'][0]))
                pages[page_index]=dict(candidates=candidates,source='local_yolo')
                del prediction
        finally:
            del detector
            gc.collect()
            if torch.cuda.is_available():torch.cuda.empty_cache()
    return pages


bubble_pages=cached_stage('contour_bubbles',
    dict(weights_sha256=BUBBLE_SHA256,settings=DETECTION,ultralytics=version('ultralytics'),
         image_mode='original_RGB',algorithm=function_digest(predict_bubbles),
         cache_discovery=function_digest(find_cached_bubbles)),predict_bubbles)
print('Bubble candidates:',[len(p['candidates']) for p in bubble_pages])


RUN   contour_bubbles
Bubble candidates: [11, 8, 7]


## 17. Concatenate surviving OCR and assign contour roles
Reuse the v2 plain-function helper. Individual normal/faint boundary checks select overlapping groups; no merged-boundary success is required. The combined YOLO crop is padded by 20%. Actual white/solid-region edges determine labels, with failed/uncertain-only faint-barrier retry and orientation-independent smooth-arc evidence. Crop borders never count as edges. Every surviving text must be emitted exactly once. No learned narration or tail override runs.


In [17]:
CONTOUR_INFO=dict(module_sha256=file_hash(CONTOUR_MODULE_PATH),
    grouping_sha256=file_hash(CONTOUR_GROUPING_PATH),check=CONTOUR_SETTINGS,
    region=REGION_SETTINGS,faint_region=FAINT_REGION_SETTINGS,group_faint=GROUP_FAINT_SETTINGS)


def contour_texts(page_index):
    texts=[];page_number=int(PAGE_FILES[page_index].stem)
    for index,decision in enumerate(ocr_selection[page_index]):
        filtered=ocr_agreement[page_index][index]
        if filtered['delete_ocr']:continue
        texts.append(dict(box_id=f'PG{page_number}:T{index+1}',text_index=index,box=decision['box'],
            text=decision['selected_text'],magi_text=decision['magi_text'],paddle_text=decision['paddle_text'],
            ocr_agreement=filtered['ocr_agreement']))
    return texts


def make_contour_roles():
    pages=[]
    for page_index,path in enumerate(PAGE_FILES):
        analysis,_=contour_model.analyze_page(bubble_pages[page_index]['candidates'],
            Image.fromarray(crop_pages[page_index]),contour_texts(page_index),int(path.stem))
        for row in analysis['rows']:
            row.update(sequence_id=SEQUENCE_ID,page_index=page_index,page_file=path.name)
        pages.append(analysis)
    return pages


contour_pages=cached_stage('contour_roles',
    dict(selector=stage_ref('ocr_selection'),filter=stage_ref('ocr_agreement'),
         bubbles=stage_ref('contour_bubbles'),contour=CONTOUR_INFO,
         algorithm=function_digest(make_contour_roles),adapter=function_digest(contour_texts)),make_contour_roles)
contour_rows=[r for page in contour_pages for r in page['rows']]
expected={r['box_id'] for r in raw_text_entities if not r['delete_ocr']}
emitted=[bid for row in contour_rows for bid in row['box_ids']]
assert len(emitted)==len(set(emitted)) and set(emitted)==expected,'Missing/duplicated surviving text'
display(pd.DataFrame(contour_rows).reindex(columns=['box_id','box_ids','text','text_role','reason','needs_review']))


RUN   contour_roles


,box_id,box_ids,text,text_role,reason,needs_review
0,PG1:T1,[PG1:T1],MANGA STREAM DO NOT MIRROR,NARRATED,no YOLO bubble matched this surviving Magi text,False
1,PG1:T2,[PG1:T2],GOT YOUR TOMATO DEVIL RIGHT HERE.,SPOKEN,white region with substantial curved sides; ov...,False
2,PG1:T3,[PG1:T3],"IT REVIVES FROM ITS SEEDS, SO YOU SHOULD BURN IT.",SPOKEN,white region with meaningful visible sharp edges,False
3,PG1:M1,"[PG1:T4, PG1:T5]","GOOD WORK, DENJI. THIS GUY'S CORPSE'LL FETCH A...",SPOKEN,white region with meaningful visible sharp edges,False
4,PG1:T6,[PG1:T6],THE REWARD'S 400 GRAND.,SPOKEN,white region with meaningful visible sharp edges,False
5,PG1:T7,[PG1:T7],Though you look at the bottom-of-the-the-the-a...,NARRATED,region may have leaked outside the bubble; unc...,True
6,PG1:T8,[PG1:T8],THANKS !!,SPOKEN,white region with meaningful visible sharp edges,False
7,PG1:T9,[PG1:T9],I'M TAKING OHT 17O GRAND tO COVER YOUR LOAN AN...,SPOKEN,white region with meaningful visible sharp edges,False
8,PG1:T10,[PG1:T10],AND THEN THERE'S THE COMMISSION FOR REFERRING ...,SPOKEN,white region with substantial curved sides; ov...,False
9,PG1:T11,[PG1:T11],ONLY 70 GRAND LEFT...,SPOKEN,white region with meaningful visible curved sides,False


## 18. Route group labels to narrator or sequence character
Narrated groups use NARRATION. Spoken groups use the single distinct entity among their original Magi text-character links. A partial link is used with review; missing or conflicting entities remain unresolved. We do not guess a new association from the combined rectangle. Preserve all member links and OCR/contour review reasons.


In [18]:
def resolve_group_speaker(role,members):
    entity_ids=list(dict.fromkeys(entity for member in members for entity in member['entity_ids']))
    missing=[m['box_id'] for m in members if not m['entity_ids']]
    links=[dict(box_id=m['box_id'],entity_ids=m['entity_ids'],character_links=m['character_links'],
                association_status=m['association_status']) for m in members]
    if role=='NARRATED':speaker,status,review='NARRATION','narrator',False
    elif len(entity_ids)>1:speaker,status,review=None,'conflicting_entities',True
    elif not entity_ids:speaker,status,review=None,'unassociated',True
    elif missing:speaker,status,review=entity_ids[0],'partially_linked',True
    else:speaker,status,review=entity_ids[0],'linked',False
    return dict(speaker=speaker,speaker_status=status,speaker_needs_review=review,
                candidate_entity_ids=entity_ids,unlinked_member_ids=missing,member_associations=links)


def make_group_speakers():
    lookup={r['box_id']:r for r in raw_text_entities if not r['delete_ocr']}
    rows=[]
    for contour in contour_rows:
        members=[lookup[bid] for bid in contour['box_ids']]
        assert all(m['page_index']==contour['page_index'] for m in members)
        row=copy.deepcopy(contour)
        row.update(resolve_group_speaker(row['text_role'],members))
        ocr_review=any(ocr_selection[m['page_index']][m['text_index']]['needs_review'] for m in members)
        row['needs_review']=bool(row['needs_review'] or row['speaker_needs_review'] or ocr_review)
        row['ocr_needs_review']=ocr_review
        rows.append(row)
    return rows


grouped_records=cached_stage('contour_speakers',
    dict(roles=stage_ref('contour_roles'),associations=stage_ref('text_entities'),filter=stage_ref('ocr_agreement'),
         algorithm=function_digest(make_group_speakers),routing=function_digest(resolve_group_speaker)),make_group_speakers)
# Export an audit of all original boxes, including deleted boxes, without reordering them.
by_member={bid:row for row in grouped_records for bid in row['box_ids']}
box_records=[]
for original in raw_text_entities:
    row=copy.deepcopy(original);group=by_member.get(row['box_id'])
    if row['delete_ocr']:
        assert group is None
        row.update(group_id=None,text_role=None,speaker=None,speaker_status='deleted')
    else:
        assert group is not None
        row.update(group_id=group['box_id'],text_role=group['text_role'],speaker=group['speaker'],
            speaker_status=group['speaker_status'],needs_review=group['needs_review'],contour_reason=group['reason'])
    box_records.append(row)
display(pd.DataFrame(grouped_records).reindex(columns=['box_id','box_ids','speaker','speaker_status','text_role','text','needs_review']))


RUN   contour_speakers


,box_id,box_ids,speaker,speaker_status,text_role,text,needs_review
0,PG1:T1,[PG1:T1],NARRATION,narrator,NARRATED,MANGA STREAM DO NOT MIRROR,False
1,PG1:T2,[PG1:T2],E3,linked,SPOKEN,GOT YOUR TOMATO DEVIL RIGHT HERE.,False
2,PG1:T3,[PG1:T3],E3,linked,SPOKEN,"IT REVIVES FROM ITS SEEDS, SO YOU SHOULD BURN IT.",False
3,PG1:M1,"[PG1:T4, PG1:T5]",E1,linked,SPOKEN,"GOOD WORK, DENJI. THIS GUY'S CORPSE'LL FETCH A...",False
4,PG1:T6,[PG1:T6],E1,linked,SPOKEN,THE REWARD'S 400 GRAND.,False
5,PG1:T7,[PG1:T7],NARRATION,narrator,NARRATED,Though you look at the bottom-of-the-the-the-a...,True
6,PG1:T8,[PG1:T8],E3,linked,SPOKEN,THANKS !!,False
7,PG1:T9,[PG1:T9],NaN,unassociated,SPOKEN,I'M TAKING OHT 17O GRAND tO COVER YOUR LOAN AN...,True
8,PG1:T10,[PG1:T10],NaN,unassociated,SPOKEN,AND THEN THERE'S THE COMMISSION FOR REFERRING ...,True
9,PG1:T11,[PG1:T11],E3,linked,SPOKEN,ONLY 70 GRAND LEFT...,False


## 19. Annotated page viewer
Yellow outlines are spoken; grey outlines are narrated. Show member OCR boxes, groups, characters/entities, original Magi association links, panels or deleted OCR independently. Speaker labels are final group assignments; character links remain the original member associations. If a region has no visible outline, show its member text rectangles.


In [19]:
def draw_box(ax,box,color,label=None,style='-'):
    a,b,c,d=box
    ax.add_patch(patches.Rectangle((a,b),c-a,d-b,fill=False,edgecolor=color,linewidth=2,linestyle=style))
    if label:ax.text(a,b,label,fontsize=8,color='black',bbox=dict(facecolor=color,alpha=.8))


def center(box):
    a,b,c,d=box;return ((a+c)/2,(b+d)/2)


def show_page(page_index=0,groups=True,member_boxes=True,characters=True,associations=False,
              panels=False,show_deleted=False,show_ocr=True,save_path=None):
    result=per_page_results[page_index]
    fig,ax=plt.subplots(figsize=(13,20));ax.imshow(crop_pages[page_index])
    if panels:
        for box in result.get('panels',[]):draw_box(ax,box,'lime')
    cmap=plt.get_cmap('tab20',max(1,len(identified_char)))
    colors={entity:cmap(i) for i,entity in enumerate(identified_char)}
    if characters:
        for i,box in enumerate(result['characters']):
            entity=char_to_entity[f'PG{page_index+1}:C{i+1}']
            draw_box(ax,box,colors[entity],f'C{i+1} / {entity}')
    if groups:
        for row in grouped_records:
            if row['page_index']!=page_index:continue
            color='yellow' if row['text_role']=='SPOKEN' else 'grey'
            if row['visible_segments']:
                for segment in row['visible_segments']:
                    points=np.asarray(segment);ax.plot(points[:,0],points[:,1],color=color,linewidth=3)
            else:
                for box in row['member_text_boxes']:draw_box(ax,box,color)
            if member_boxes:
                for bid,box in zip(row['box_ids'],row['member_text_boxes']):draw_box(ax,box,color,bid,style='--')
            label=row['box_id']+' '+row['text_role']+' / '+(row['speaker'] or row['speaker_status'])
            if row['needs_review']:label+=' [review]'
            if show_ocr:label+=' | '+row['text'][:90]
            a,b,_,_=row['text_box'];ax.text(a,b,label,fontsize=8,bbox=dict(facecolor=color,alpha=.8))
    if associations:
        for record in raw_text_entities:
            if record['page_index']!=page_index or record['delete_ocr']:continue
            x,y=center(record['box'])
            for link in record['character_links']:
                u,v=center(result['characters'][link['character_index']])
                ax.plot([x,u],[y,v],color=colors[link['entity_id']],linewidth=1,alpha=.8)
    if show_deleted:
        for row in box_records:
            if row['page_index']==page_index and row['delete_ocr']:
                draw_box(ax,row['box'],'red','DELETED '+row['box_id'],style='--')
    ax.set_title(f'{PAGE_FILES[page_index].name} | yellow: spoken | grey: narrated | red: deleted')
    ax.axis('off');fig.tight_layout()
    if save_path:fig.savefig(save_path,bbox_inches='tight',dpi=150)
    plt.show();return fig


## 20. Inspect a group: original crop, region, faint barriers and visible edges
This viewer recomputes only CPU contour checks for the selected page and holds masks in memory. It does not reload or run models. The table lists each member's original character/entity candidates, plus the final speaker and arc metrics. In a disconnected group, inspect every member attempt and its region decision.


In [20]:
inspection_pages={}


def inspect_group(group_id):
    row=next(r for r in grouped_records if r['box_id']==group_id)
    page_index=row['page_index']
    if page_index not in inspection_pages:
        analysis,debug=contour_model.analyze_page(bubble_pages[page_index]['candidates'],
            Image.fromarray(crop_pages[page_index]),contour_texts(page_index),int(PAGE_FILES[page_index].stem))
        inspection_pages[page_index]=dict(analysis=analysis,debug=debug)
    cached=inspection_pages[page_index]
    index=next(i for i,r in enumerate(cached['analysis']['rows']) if r['box_id']==group_id)
    print(group_id,row['text_role'],'|',row['speaker'] or row['speaker_status'],'| Review:',row['needs_review'])
    print(row['text']);print('Members:',row['box_ids']);print('Reason:',row['reason'])
    for attempt in cached['debug'][index]['attempts']:
        checked=attempt['result'];region=checked['best'];left,top,_,_=checked['bounds']
        fig,axes=plt.subplots(1,4,figsize=(18,6))
        axes[0].imshow(checked['crop'])
        for a,b,c,d in row['member_text_boxes']:
            axes[0].add_patch(patches.Rectangle((a-left,b-top),c-a,d-b,fill=False,edgecolor='deepskyblue'))
        axes[0].set_title(f"YOLO {attempt['candidate_id']} + 20% per side")
        axes[1].imshow(region['mask'] if region else np.zeros(checked['crop'].shape[:2]),cmap='gray')
        axes[1].set_title('White / solid background region')
        axes[2].imshow(checked.get('retry_barrier',checked.get('barrier',np.zeros(checked['crop'].shape[:2]))),cmap='gray')
        axes[2].set_title('Faint-edge barriers (retry only)')
        overlay=checked['crop'].copy()
        if region:
            points=region['points']
            for i in range(len(points)):
                nxt=(i+1)%len(points)
                if not (region['visible'][i] and region['visible'][nxt]):continue
                color=(255,220,0) if region['curved'][i] else (0,200,255) if region['flat'][i] else (255,140,0)
                cv2.line(overlay,tuple(points[i]),tuple(points[nxt]),color,2)
            print('Region:',region['text_role'],region['reason'],'| Pass:',region['extraction_pass'])
        axes[3].imshow(overlay);axes[3].set_title('Curves yellow | flat cyan | sharp orange')
        print('Retry:',checked['retry_attempted'],'| First:',checked['first_pass'])
        if checked.get('retry_reason'):print('Retry result:',checked['retry_reason'])
        for ax in axes:ax.axis('off')
        fig.tight_layout();plt.show()
    display(pd.DataFrame(row['member_associations']))
    display(pd.DataFrame([{k:v for k,v in row.items() if k not in ['contour_points','visible_segments','member_associations']}]))


if widgets is not None:
    page_picker=widgets.Dropdown(options=[(path.name,i) for i,path in enumerate(PAGE_FILES)],description='Page:')
    display(widgets.interactive(show_page,page_index=page_picker,groups=True,member_boxes=True,
        characters=True,associations=False,panels=False,show_deleted=False,show_ocr=True,
        save_path=widgets.fixed(None)))
    if grouped_records:
        group_picker=widgets.Dropdown(options=[(r['box_id']+' '+r['text'][:45],r['box_id']) for r in grouped_records],description='Group:')
        display(widgets.interactive(inspect_group,group_id=group_picker))
else:
    show_page(0)
    if grouped_records:inspect_group(grouped_records[0]['box_id'])
# Direct calls are also available: show_page(0,associations=True), inspect_group('PG1:M1').


interactive(children=(Dropdown(description='Page:', options=(('01.png', 0), ('02.png', 1), ('03.png', 2)), val…

interactive(children=(Dropdown(description='Group:', options=(('PG1:T1 MANGA STREAM DO NOT MIRROR', 'PG1:T1'),…

## 21. Export the labels.jsonl transcript and detailed audits
transcript.jsonl contains one object with sequence_id and pages; each page contains speaker/text rows only. Narration uses NARRATION, resolved entities use stable characterN names, and unresolved speech uses UNKNOWN. Original page slots are preserved; unprocessed pages in single-page mode are empty lists. Predicted characterN IDs do not assert a match to GT identities. Detailed role/speaker/review audits remain in separate exports.


In [ ]:
def build_transcript(sequence_id,records,page_files,sequence_files,entity_ids):
    # Stable sequence-local names; these are predicted identities, not GT character matches.
    ordered_entities=sorted(entity_ids,key=lambda eid:(int(eid[1:]) if eid.startswith('E') and eid[1:].isdigit() else float('inf'),eid))
    names={entity:f'character{i+1}' for i,entity in enumerate(ordered_entities)}
    page_positions={path.name:i for i,path in enumerate(sequence_files)}
    pages=[[] for _ in sequence_files]
    for row in sorted(records,key=lambda r:(r['page_index'],r['text_index'])):
        page=page_positions[page_files[row['page_index']].name]
        speaker='NARRATION' if row['text_role']=='NARRATED' else names.get(row['speaker'],'UNKNOWN')
        pages[page].append(dict(speaker=speaker,text=row['text']))
    return dict(sequence_id=sequence_id,pages=pages),names


# Keep detailed exports; the transcript below has exactly the labels.jsonl contract.
sequence_files=sorted(path for path in SEQUENCE_ROOT.glob('*.png') if path.stem.isdigit())
transcript,entity_to_character=build_transcript(SEQUENCE_ID,grouped_records,PAGE_FILES,sequence_files,identified_char)
write_json(OUTPUT_DIR/'transcript.json',transcript)
# One sequence object per physical line. Replace this run's file rather than appending duplicates.
(OUTPUT_DIR/'transcript.jsonl').write_text(json.dumps(transcript,ensure_ascii=False)+'\n',encoding='utf-8')
write_json(OUTPUT_DIR/'transcript_speaker_mapping.json',entity_to_character)


dialogue_pages=[[] for _ in PAGE_FILES]
for row in grouped_records:
    dialogue_pages[row['page_index']].append(dict(group_id=row['box_id'],box_ids=row['box_ids'],
        speaker=row['speaker'],speaker_status=row['speaker_status'],text_role=row['text_role'],
        text=row['text'],needs_review=row['needs_review']))
write_json(OUTPUT_DIR/'dialogue_pages.json',dialogue_pages)
write_json(OUTPUT_DIR/'group_predictions.json',grouped_records)
pd.DataFrame(grouped_records).to_csv(OUTPUT_DIR/'group_predictions.csv',index=False)
write_json(OUTPUT_DIR/'box_predictions.json',box_records)
pd.DataFrame(box_records).to_csv(OUTPUT_DIR/'box_predictions.csv',index=False)
deleted=[r for r in box_records if r['delete_ocr']]
write_json(OUTPUT_DIR/'deleted_ocr.json',deleted)
pd.DataFrame(deleted).to_csv(OUTPUT_DIR/'deleted_ocr.csv',index=False)
write_json(OUTPUT_DIR/'grouping_audit.json',[p['grouping'] for p in contour_pages])
write_json(OUTPUT_DIR/'entities.json',entity_results)
write_json(OUTPUT_DIR/'run_metadata.json',dict(sequence_id=SEQUENCE_ID,pages=PAGE_METADATA,
    narration_method='contour_narrator2',contour=CONTOUR_INFO,bubble_weights_sha256=BUBBLE_SHA256,
    bubble_settings=DETECTION,character_checkpoint=CHECKPOINT_INFO,entity_threshold=ENTITY_THRESHOLD,
    association_head=str(ASSOCIATION_HEAD_PATH) if ASSOCIATION_HEAD_PATH else 'original_magi',
    tail_override=False,ocr_filter=dict(agreement_delete_below=OCR_AGREEMENT_DELETE_BELOW,delete_sfx=True,clutter_phrases=CLUTTER_PHRASES,
        clutter_threshold=CLUTTER_FUZZY_THRESHOLD),
    manifest=str(MANIFEST_PATH),stages=manifest['stages']))
lines=[]
for path,page in zip(sequence_files,transcript['pages']):
    lines.append(f'[Page {path.name}]')
    lines.extend(f"{row['speaker']}: {row['text']}" for row in page)
    lines.append('')
(OUTPUT_DIR/'transcript.txt').write_text('\n'.join(lines),encoding='utf-8')

display(pd.DataFrame([dict(page=path.name,detected=len(magi_raw_results[i]['texts']),
    deleted=sum(r['delete_ocr'] for r in box_records if r['page_index']==i),
    output_rows=len(dialogue_pages[i]),spoken=sum(r['text_role']=='SPOKEN' for r in dialogue_pages[i]),
    narrated=sum(r['text_role']=='NARRATED' for r in dialogue_pages[i]),
    review=sum(r['needs_review'] for r in dialogue_pages[i])) for i,path in enumerate(PAGE_FILES)]))
print('Saved:',OUTPUT_DIR)
# Optional annotated page export (no model work):
# for i,path in enumerate(PAGE_FILES):
#     figure=show_page(i,save_path=OUTPUT_DIR/f'annotated_{path.stem}.png')
#     plt.close(figure)


## 22. Clean each transcript entry with local Gemma 3 4B IT
Uses the approved few-shot prompt, one text entry at a time, after all OCR grouping and speaker decisions. The local model loads once in 4-bit on cache misses and is released afterward. Raw transcripts are preserved as transcript_raw files; final transcript files contain corrected text. Speakers, page positions, entry counts and labels.jsonl schema stay fixed. Empty responses fall back to the original; every change is audited. Detailed group/box/dialogue files retain their original OCR.


In [22]:
# This final stage changes transcript text only, after grouping and speaker assignment.
GEMMA_MODEL_PATH=Path('C:/MODELS/gemma-3-4b-it')
GEMMA_EXAMPLES=[
    {'role': 'system', 'content':
     'You restore English manga dialogue corrupted by OCR. '
     'Repair obvious broken words. Preserve legitimate names, valid single-letter words such as I and A, '
     'Preserve Punctutaion and Punctuation Only Sentences'
     'Remove nonsensical inserted letter sequences and accidental isolated letters that interrupt a sentence. '
     'capitalization, meaning, punctuation and unfinished sentences. '
     'Do not rewrite the dialogue, complete it or answer it. '
     'any single character repeated multiple times could be punctuations mis interpreted '
     'Return only the restored text, with no explanation or quotation marks.'},
    {'role': 'user', 'content': 'PLEASE Q OPEN THE WINDOW.'},
    {'role': 'assistant', 'content': 'PLEASE OPEN THE WINDOW.'},
    {'role': 'user', 'content': 'I CAN HEAR XZQWZXQ SOMEONE OUTSIDE...'},
    {'role': 'assistant', 'content': 'I CAN HEAR SOMEONE OUTSIDE...'},
    {'role': 'user', 'content': 'AH... SHIROGANE, WAIT!'},
    {'role': 'assistant', 'content': 'AH... SHIROGANE, WAIT!'},
    {'role': 'user', 'content': '.....'},
    {'role': 'assistant', 'content': '.....'},
    {'role': 'user', 'content': 'PLEASE S OPEN THE WINDOW.'},
    {'role': 'assistant', 'content': 'PLEASE OPEN THE WINDOW.'},
]
FORCE.setdefault('gemma_transcript',False)
STAGE_VERSIONS['gemma_transcript']=1
# Always rebuild from original group OCR, so rerunning cleanup cannot repeatedly edit its own output.
raw_transcript,_=build_transcript(SEQUENCE_ID,grouped_records,PAGE_FILES,sequence_files,identified_char)
# Keep the exact original OCR transcript before changing any final transcript exports.
write_json(OUTPUT_DIR/'transcript_raw.json',raw_transcript)
(OUTPUT_DIR/'transcript_raw.jsonl').write_text(json.dumps(raw_transcript,ensure_ascii=False)+'\n',encoding='utf-8')
raw_lines=[]
for path,page in zip(sequence_files,raw_transcript['pages']):
    raw_lines.append(f'[Page {path.name}]')
    raw_lines.extend(f"{row['speaker']}: {row['text']}" for row in page)
    raw_lines.append('')
(OUTPUT_DIR/'transcript_raw.txt').write_text('\n'.join(raw_lines),encoding='utf-8')
assert (GEMMA_MODEL_PATH/'config.json').is_file(),f'Missing local Gemma: {GEMMA_MODEL_PATH}'
GEMMA_IDENTITY=dict(path=str(GEMMA_MODEL_PATH),
    files={path.name:dict(size=path.stat().st_size,mtime_ns=path.stat().st_mtime_ns,
                         sha256=file_hash(path) if path.suffix=='.json' else None)
           for path in sorted(GEMMA_MODEL_PATH.iterdir())
           if path.is_file() and path.suffix in {'.json','.safetensors','.bin','.model'}})


def correct_transcript_text(text,model,tokenizer):
    if not text.strip():return text
    messages=GEMMA_EXAMPLES+[{'role':'user','content':text}]
    inputs=tokenizer.apply_chat_template(messages,add_generation_prompt=True,
        tokenize=True,return_dict=True,return_tensors='pt').to(model.device)
    with torch.inference_mode():
        output=model.generate(**inputs,do_sample=False,
            max_new_tokens=max(256,inputs['input_ids'].shape[1]*2))
    return tokenizer.decode(output[0,inputs['input_ids'].shape[1]:],skip_special_tokens=True).strip()


def make_gemma_transcript():
    cleaned=copy.deepcopy(raw_transcript);audit=[]
    if not any(row['text'].strip() for page in cleaned['pages'] for row in page):
        return dict(transcript=cleaned,audit=audit)
    from transformers import AutoTokenizer,Gemma3ForConditionalGeneration,BitsAndBytesConfig
    assert torch.cuda.is_available(),'Gemma cleanup requires your CUDA godfather.venv kernel.'
    compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    tokenizer=AutoTokenizer.from_pretrained(str(GEMMA_MODEL_PATH),local_files_only=True)
    model=Gemma3ForConditionalGeneration.from_pretrained(str(GEMMA_MODEL_PATH),local_files_only=True,
        device_map='auto',dtype=compute_dtype,
        quantization_config=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
                                             bnb_4bit_compute_dtype=compute_dtype)).eval()
    try:
        # One transcript entry at a time; never send speaker IDs to the correction model.
        seen={}
        for page_index,page in enumerate(cleaned['pages']):
            for row_index,row in enumerate(page):
                original=row['text']
                if original not in seen:seen[original]=correct_transcript_text(original,model,tokenizer)
                answer=seen[original]
                row['text']=answer if answer else original
                audit.append(dict(page_index=page_index,row_index=row_index,speaker=row['speaker'],
                    raw_text=original,cleaned_text=row['text'],changed=row['text']!=original,
                    empty_output_fallback=bool(original.strip() and not answer)))
    finally:
        del model,tokenizer
        gc.collect()
        torch.cuda.empty_cache()
    return dict(transcript=cleaned,audit=audit)


# A compatible cache skips Gemma loading and generation on subsequent runs.
gemma_cleanup=cached_stage('gemma_transcript',dict(input_transcript=digest(raw_transcript),
    model=GEMMA_IDENTITY,prompt=GEMMA_EXAMPLES,quantization='nf4',generation='greedy_dynamic_max_tokens_v1',
    transformers=version('transformers'),bitsandbytes=version('bitsandbytes'),
    algorithm=function_digest(make_gemma_transcript),corrector=function_digest(correct_transcript_text)),make_gemma_transcript)
cleaned_transcript=gemma_cleanup['transcript']
assert cleaned_transcript['sequence_id']==raw_transcript['sequence_id']
assert len(cleaned_transcript['pages'])==len(raw_transcript['pages'])
for original_page,cleaned_page in zip(raw_transcript['pages'],cleaned_transcript['pages']):
    assert len(original_page)==len(cleaned_page)
    assert [r['speaker'] for r in original_page]==[r['speaker'] for r in cleaned_page]
    assert all(set(r)=={'speaker','text'} for r in cleaned_page)
# Final transcript filenames now contain the cleaned text in the original labels.jsonl schema.
transcript=copy.deepcopy(cleaned_transcript)
write_json(OUTPUT_DIR/'transcript.json',transcript)
(OUTPUT_DIR/'transcript.jsonl').write_text(json.dumps(transcript,ensure_ascii=False)+'\n',encoding='utf-8')
lines=[]
for path,page in zip(sequence_files,transcript['pages']):
    lines.append(f'[Page {path.name}]')
    lines.extend(f"{row['speaker']}: {row['text']}" for row in page)
    lines.append('')
(OUTPUT_DIR/'transcript.txt').write_text('\n'.join(lines),encoding='utf-8')
write_json(OUTPUT_DIR/'transcript_cleanup_audit.json',gemma_cleanup['audit'])
# Detailed group/box/dialogue exports retain original OCR; cleaned text is in transcript files above.
metadata=json.loads((OUTPUT_DIR/'run_metadata.json').read_text(encoding='utf-8'))
metadata['transcript_cleanup']=dict(model=GEMMA_IDENTITY,stage=stage_ref('gemma_transcript'),
    changed_entries=sum(row['changed'] for row in gemma_cleanup['audit']),raw_export='transcript_raw.jsonl')
metadata['stages']=manifest['stages']
write_json(OUTPUT_DIR/'run_metadata.json',metadata)
print('Gemma cleanup:',len(gemma_cleanup['audit']),'entries;',sum(r['changed'] for r in gemma_cleanup['audit']),'changed')


RUN   gemma_transcript


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Gemma cleanup: 25 entries; 8 changed


## 23. Print the cleaned sequence transcript
The final schema-compatible transcript is printed after Gemma cleanup. Original OCR is preserved in transcript_raw.jsonl and the cleanup audit.


In [23]:
# Print the cleaned transcript in the development/labels.jsonl structure.
print(json.dumps(transcript,ensure_ascii=False,indent=2))


{
  "sequence_id": "seq_2fae297a18af2a57",
  "pages": [
    [
      {
        "speaker": "NARRATION",
        "text": "MANGA STREAM DO NOT MIRROR"
      },
      {
        "speaker": "character3",
        "text": "GOT YOUR TOMATO DEVIL RIGHT HERE."
      },
      {
        "speaker": "character3",
        "text": "IT REVIVES FROM ITS SEEDS, SO YOU SHOULD BURN IT."
      },
      {
        "speaker": "character1",
        "text": "GOOD WORK, DENJI. THIS GUY'S CORPSE'LL FETCH A PRETTY PENNY ON THE BLACK MARKET."
      },
      {
        "speaker": "character1",
        "text": "THE REWARD'S 400 GRAND."
      },
      {
        "speaker": "NARRATION",
        "text": "Though you look at the bottom of the art."
      },
      {
        "speaker": "character3",
        "text": "THANKS!!"
      },
      {
        "speaker": "UNKNOWN",
        "text": "I’M TAKING TO GRAND TO COVER YOUR LOAN AND INTEREST PAYMENTS."
      },
      {
        "speaker": "UNKNOWN",
        "text": "AND THEN THERE'